In [1]:
import casadi as cs
import numpy as np
from time import perf_counter
from matplotlib import pyplot as plt
import pickle
import ctypes
import os

import problem_definition
from real_time import Real_Time
from solver_banks import Solver 
from OCP_layout import Layout
from dynamics import Hopper 
import LUT_helper as helper

In [2]:
def initialize_mode(LUT, count, E0, phi_end=1.0):
    return np.column_stack([
        np.asarray(LUT([E0, phi])[:-1]).squeeze()
        for phi in np.linspace(0, phi_end, count)
    ])

In [3]:
def alternating(mode,n,dynamics):
    return tuple(mode if k%2 == 0 else dynamics.next_mode[mode] for k in range(n))

def candidate_sequences(mode, nominal, dynamics, max_modes=4):
    counts = {
        "nominal": nominal, "minus": nominal - 1, "plus": nominal + 1,
    }
    return {
        name: alternating(mode, n, dynamics) for name, n in counts.items() if 1 <= n <= max_modes
    }

def select_candidate(candidates,vilation_limit):
    accepted = []
    for candidate in candidates:
        if candidate["success"] and candidate["violation"] <= vilation_limit:
            accepted.append(candidate)
    return min(accepted, key=lambda c: c["merit"]) if accepted else None

In [4]:
def resample(y, start, end, n):
    old_grid = np.linspace(0.0, 1.0, y.shape[-1])
    new_grid = np.linspace(start, end, n)

    return np.array([
        np.interp(new_grid, old_grid, row)
        for row in np.atleast_2d(y)
    ]).squeeze()

def resample_timed(x, time, count):
    grid = np.linspace(time[0], time[-1], count)
    return np.vstack([
        np.interp(grid, time, row)
        for row in np.asarray(x)
    ])

In [5]:
def shift_trajectory(solution, x0, elapsed):
    idx_m = 0
    while elapsed >= solution["t"][idx_m]:
        elapsed -= solution["t"][idx_m]
        idx_m += 1

    modes = solution["modes"][idx_m:] 
    x = [xi.copy() for xi in solution["x"][idx_m:]]
    u = [ui.copy() for ui in solution["u"][idx_m:]]
    t = solution["t"][idx_m:].copy()
    fraction = elapsed / t[0]
    x[0] = resample(x[0], fraction, 1.0, x[0].shape[1])
    u[0] = resample(u[0], fraction, 1.0, u[0].shape[-1])
    t[0] -= elapsed

    x[0][:, 0] = np.asarray(x0).squeeze()

    
    shifted = {
        "modes": modes,
        "x": x,
        "u": u,
        "t": t,
        "E": solution["E"],
        "phi": solution["phi"],
    }

    return shifted, len(modes)

In [6]:
## Verified chat-gpt generated code

def warm_candidate(shifted, sequence, T):
    """Construct the nominal or nominal-minus-one candidate."""
    count = len(sequence)

    candidate = {
        "modes": sequence,
        "x": [x.copy() for x in shifted["x"][:count]],
        "u": [u.copy() for u in shifted["u"][:count]],
        "t": shifted["t"][:count].copy(),
        "E": shifted["E"],
        "phi": shifted["phi"],
    }

    if count < len(shifted["modes"]):
        # The new terminal mode previously ended at an event. Remove its
        # event node so that the terminal trajectory ends inside the mode.
        n_state = candidate["x"][-1].shape[-1]
        end = (n_state - 2) / (n_state - 1)
        candidate["x"][-1] = resample(candidate["x"][-1], 0.0, end, n_state)
        candidate["u"][-1] = resample(candidate["u"][-1], 0.0, end,candidate["u"][-1].shape[-1])

    # The final mode absorbs the missing part of the fixed horizon.
    candidate["t"][-1] = T - sum(candidate["t"][:-1])

    return candidate

In [7]:
## WARNING chat-gpt generated

def passive_rollout(x0,m,dynamics,duration):
    x = np.zeros((dynamics.dim[m],dynamics.N[m]))
    x[:,0] = np.asarray(x0).squeeze()
    h = float(duration) / (dynamics.N-1)
    for k in range(dynamics.N-1):
        x[:,k+1] = np.array(dynamics.rk4[m](x[:,k],h,0.0)).squeeze()
    return x


def passive_candidate(shifted, sequence, T,dynamics, terminal_duration=1e-2):
    x = [xi.copy() for xi in shifted["x"]]
    u = [ui.copy() for ui in shifted["u"]]
    duration = shifted["t"].copy()

    current_mode = shifted["modes"][-1]
    next_mode = sequence[-1]
    remaining = T - sum(duration)

    # Continue passively from the end of the warm trajectory to the event.
    event_time, event_trajectory = dynamics.integrate_until_event(x[-1][:, -1], current_mode)
    event_time = event_time.squeeze()
    event_trajectory = np.asarray(event_trajectory)
    passive_duration = float(event_time[-1])

    # Leave a positive duration for the added terminal mode.
    declared_duration = min(passive_duration, remaining - terminal_duration)

    # Retain the old final-mode trajectory and append its passive continuation.
    old_duration = duration[-1]
    old_time = np.linspace(0.0, old_duration, x[-1].shape[-1])
    tail_time = (old_duration + declared_duration * event_time[1:] / passive_duration)

    combined_time = np.concatenate((old_time, tail_time))
    combined_state = np.column_stack((x[-1], event_trajectory[:, 1:]))

    x[-1] = resample_timed(combined_state,combined_time,dynamics.N[current_mode])

    # Preserve the old controls and use zero control over the passive extension.
    control_time = np.linspace(0.0, old_duration, u[-1].size, endpoint=False)
    new_control_time = np.linspace(0.0, old_duration + declared_duration, dynamics.N[current_mode])[:-1]
    u[-1] = np.interp(
        new_control_time,
        np.concatenate((control_time, [old_duration, old_duration + declared_duration])),
        np.concatenate((u[-1], [0.0, 0.0])),
    )

    duration[-1] = old_duration + declared_duration

    # Reset at the event and propagate the newly added mode.
    state = np.asarray(dynamics.convert[current_mode](event_trajectory[:, -1])).squeeze()

    final_duration = T - sum(duration)
    x.append(passive_rollout(state, next_mode, dynamics, final_duration,))
    u.append(np.zeros(dynamics.N[next_mode] - 1))
    duration.append(final_duration)

    return {
        "modes": sequence,
        "x": x,
        "u": u,
        "t": duration,
        "E": shifted["E"],
        "phi": shifted["phi"],
    }

In [8]:
def qp_data(w,p,linearize,con_min,con_max,w_min,w_max,delta):
    grad,H,A,residual = linearize(w,p)
    nominal = np.asarray(w).squeeze()
    return {
        "h": H, "g": grad, "a": A,
        "lbx": np.maximum(w_min-nominal, -delta),
        "ubx": np.minimum(w_max-nominal, delta),
        "lba": con_min-residual, "uba": con_max-residual,
    }

In [9]:
def RTI_candidate(solver, seed, x0, T, mu):
    p = cs.vertcat(x0,T)
    w_bar = solver.layout.pack(seed["x"],seed["u"],seed["t"],seed["E"],seed["phi"])
    qp = qp_data(
        w_bar, p, solver.linearize,
        solver.con_min, solver.con_max,
        solver.w_min, solver.w_max, solver.delta
    )
    qp["x0"] = cs.DM.zeros(solver.layout.nw) # useless ?

    result = solve_qp(qp,solver.qp_solver)
    if not solver.qp_solver.stats()["success"]:
        return {"success" : False}

    w = w_bar + result["x"] # alpha = 1
    cost,residual = solver.values(w,p)

    violation = np.maximum(solver.con_min - residual,0) + np.maximum(residual - solver.con_max,0)
    violation_sum = float(cs.sum1(violation))
    return {
        "success": True,
        "w": w,
        "trajectory": solver.layout.unpack(w),
        "cost": float(cost),
        "violation": float(cs.mmax(violation)),
        "merit" : float(cost) + mu * violation_sum
    }

In [10]:
libc = ctypes.CDLL(None)
def solve_qp(qp,solver):
    # wrapper to remove unwanted output from hpipm
    stdout, null = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    try:  # CasADi 3.7.2's HPIPM plugin prints directly to C stdout
        os.dup2(null, 1)
        return solver(**qp)
    finally:
        libc.fflush(None)
        os.dup2(stdout, 1)
        os.close(stdout)
        os.close(null)

In [11]:
data = pickle.load(open('../data/dataset_2/dataset_21.pkl','rb'))
K = data["K"]
W = data["W"]
Nf = data["Nf"]
Ns = data["Ns"]
locomotion_NMM = np.array(data["locomotion"][2])
switching_NMM = np.array(data["switch"][2])
Emin = max(min(locomotion_NMM[:,-1]),min(switching_NMM[:,-1]))
Emax = min(max(locomotion_NMM[:,-1]),max(switching_NMM[:,-1]))

dynamics = Hopper(K,W,Nf=15,Ns=20)


In [12]:
E_grid = np.linspace(Emin, Emax, 20)
locomotion_NMM_sampled = helper.energy_sample(locomotion_NMM,E_grid)
switching_NMM_sampled = helper.energy_sample(switching_NMM,E_grid)

phi_grid = np.linspace(0,1,20)
ss_loc,sf_loc = helper.surface(locomotion_NMM_sampled,phi_grid,dynamics)
ss_swi,sf_swi = helper.surface(switching_NMM_sampled,phi_grid,dynamics)

values_sloc = ss_loc.ravel(order="C")
values_floc = sf_loc.ravel(order="C")
values_sswi = ss_swi.ravel(order="C")
values_fswi = sf_swi.ravel(order="C")
LUT_sloc = cs.interpolant('LUT_sloc', 'bspline',[E_grid,phi_grid],values_sloc)
LUT_floc = cs.interpolant('LUT_floc', 'bspline',[E_grid,phi_grid],values_floc)
LUT_sswi = cs.interpolant('LUT_sswi', 'bspline',[E_grid,phi_grid],values_sswi)
LUT_fswi = cs.interpolant('LUT_fswi', 'bspline',[E_grid,phi_grid],values_fswi)

In [13]:
E0 = 4
E_inf = max(Emin,E0-0.2)
E_sup = min(Emax,E0+0.2)

x_bounds = {
    "f" : {1: (0.0, np.inf)}, 
    "s" : {}
}
u_bounds = {
    "f" : (-1,1), 
    "s" : (-1,1)
}
t_bounds = {
    "f": (1e-2, 10.0),
    "s": (1e-2, 10.0),
}
E_bounds = (E_inf,E_sup)
phi_bounds = (0.01,0.99)

violation_limit = 1e-2
bounds = (x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)

In [14]:
sequence_variants = [alternating(m,n,dynamics) for m in 'fs' for n in range(1,5)]
targets = {
    'locomotion': {'f':LUT_floc,'s':LUT_sloc},
    'switching': {'f':LUT_fswi,'s':LUT_sswi}
}
solvers = {
    (name,seq): Solver(LUT[seq[-1]],dynamics,seq,bounds)
    for name,LUT in targets.items() for seq in sequence_variants
}

In [15]:
# Initial passive trajectory: one period + half a stance phase
target_name = "switching"
sequence = ("s", "f", "s")
phi_terminal = 0.5

ts = float(LUT_sloc([E0, 1])[-1])
tf = float(LUT_floc([E0, 1])[-1])
T = ts + tf + phi_terminal * ts

successful = {
    "modes": sequence,
    "x": [
        initialize_mode(LUT_sloc, dynamics.N["s"], E0),
        initialize_mode(LUT_floc, dynamics.N["f"], E0),
        initialize_mode(LUT_sloc, dynamics.N["s"], E0, phi_terminal),
    ],
    "u": [np.zeros(dynamics.N[m] - 1) for m in sequence],
    "t": np.array([ts, tf, phi_terminal * ts]),
    "E": E0,
    "phi": phi_terminal,
}

hopper = Real_Time(dynamics, successful["x"][0][:, 0], sequence[0])
elapsed, Ts, mu = 0.0, 5e-2, 2000

In [17]:
controller_running = True
iteration = 0
while controller_running and iteration <= 10:

    x0,m = hopper.measure()

    shifted,nominal = shift_trajectory(successful, x0, elapsed)

    sequences = candidate_sequences(m, nominal, dynamics)

    seeds = {}
    if "nominal" in sequences:
        seeds["nominal"] = warm_candidate(shifted, sequences["nominal"],T)
    if "minus" in sequences:
        seeds["minus"] = warm_candidate(shifted, sequences["minus"],T)
    if "plus" in sequences:
        seeds["plus"] = passive_candidate(shifted, sequences["plus"],T,dynamics)

    candidates = []
    for name,seq in sequences.items():
        solver = solvers[target_name, seq]
        candidate = RTI_candidate(solver, seeds[name], x0, T, mu)
        candidate["name"] = name
        candidate["sequence"] = seq
        candidates.append(candidate)

    selected = select_candidate(candidates,violation_limit)
    if selected is not None:
        u = selected["trajectory"]["u"][0][0]
        successful = selected["trajectory"]
        elapsed = Ts 
    else: 
        u = shifted["u"][0][0]
        elapsed += Ts 

    hopper.apply_control(u, Ts)
    iteration += 1

TypeError: unsupported operand type(s) for -: 'dict' and 'int'